Generate a figure like figure 8 of Blouin et al. 2019 (Paper III) in which the number abundances of each element relative to calcium are compared to other objects with all objects plotted in a column corresponding to the given element.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs
import nuclear_cross_sections as nc




#print(os.getcwd())

1730749277.027722
two_arm_compare_SDSS
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


Now I need to choose the elements that will appear in the plot. I could import an element list and check against it for ones that have a match... but I'd rather just manually list out the elements it should be checking for instead

In [2]:
base_el='Ca'
el_list=['Li','Na','Mg','K','Cr','Fe'] #ordered by atomic number
#el_list=['Na','K','Li','Cr','Fe','Mg']
#el_list=['Na','K','Li','Cr','Fe','Mg'] #ordered in decreasing volatility i.e. most volatile first.
ALi_BBN=2.72 #Coc 2014 I believe




el_space=2
arrow_alpha=0.4
arrow_head_length_mult=1.5 #pyplot default arrow head length multiple of the head_width


ssp=False
#text_y_pos=0.5
#text_va='bottom'
#text_va='center'
#text_y_pos=1.0
text_va='top'
text_y_pos=2.6
props=dict(boxstyle='round,pad=0.2',
           fc='w',
           ec='None')

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/z_plots_for_Hollands/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [5]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
#wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
#wd_abund_file='20211112_all_wd_abundances_beryllium_objects_partially_added.csv'
#wd_abund_file='20220222_all_wd_abundances_newMC_ages.csv'
#wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
lodders_abund_file='Lodders2020_solarsystem_abundances.csv'
#solar_system_object_file='solar_system_body_abundances.csv'
#solar_system_object_file='solar_system_body_abundances_mgfe_fixed.csv'
#solar_system_object_file='solar_system_body_abundances_sea_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_show_name_added.csv'
solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'
crust_file='20220304_continental_crust_vals_only.csv'

wd_num_abund_file='20220204_beryllium_WD_linear_number_abundances.csv'

#pop_abund_file='20230122_SAGA_binned_dex_abunds_MSonly.csv'
#pop_abund_file='20230122_SAGA_binned_dex_abunds_MSonly_CaHadded.csv'
#pop_abund_file='20230207_SAGA_binned_dex_abunds_MSonly_fixerrorbars.csv'
#pop_abund_file='20230207_SAGA_binned_dex_abunds_MSonly_convert_to_absolute_fixerrorbars_LiCa_calc.csv'
pop_abund_file='20241104_SAGA_binned_dex_abunds_MSonly_colorchange.csv'

In [6]:
if ssp:
    #wd_abund_file='20220304_WD_SSP_abundances.csv'
    #wd_abund_file='20221109_WD_SSP_abundances_J1636recovered.csv'
    wd_abund_file='20230207_WD_SSP_abundances_noJ2356.csv'
    show_dp=True
else:
    #wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
    #wd_abund_file='20221109_all_wd_abundances_7030MCages_DR3kinematics_missingabundsJ1636added.csv'
    wd_abund_file='20230207_all_wd_abundances_noJ2356.csv'
    show_dp=False
    
dp_arrow_file='20221110_WD_DP_abundances_arrowlengths.csv'



In [7]:
print(os.getcwd())

print(wd_abund_file)
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
dp_arrow_table=Table.read(dp_arrow_file)
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
crust_table=Table.read(crust_file)
pop_table=Table.read(pop_abund_file)
pop_table=spt.clean_color_string(pop_table,color_header='plot_color')
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')
dp_arrow_table.add_index('name')

#wd_num_abund_table=Table.read(wd_num_abund_file)
#wd_num_abund_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit



/Users/BenKaiser/Desktop/radial_velocity_calculations
20230207_all_wd_abundances_noJ2356.csv


In [8]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]
#use_wd_indices=np.where((wd_abund_table['show']==1)and (wd_abund_table['show_geo']==1))
use_wd_indices=np.where(wd_abund_table['show_geo']==1)
use_wd_abund_table=wd_abund_table[use_wd_indices]
use_wd_indices=np.where(use_wd_abund_table['show']==1)
use_wd_abund_table=use_wd_abund_table[use_wd_indices]


wd_num_abund_table=Table.read(wd_num_abund_file)
wd_num_abund_table.add_index('name')

use_pop_indices=np.where(pop_table['show']==1)
use_pop_table=pop_table[use_pop_indices]

In [9]:
spall_n_sigma=1.



t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
star_marker='o'
met_color='#1ca1f2'
met_size=3
#ci_size=6 #before adding the [Fe/H] points
ci_size=5
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=False
default_cross_source='read' #source of nuclear cross-sections for calculation of implied spallation levels



In [10]:
def get_BeCa(wd_name='GALEXJ2339-0424',n_sigma=1.):
    BeCa,BeCa_lo, BeCa_hi= acorr.get_el1el2_full_err(wd_num_abund_table.loc[wd_name],'be','ca',n_sigma=n_sigma)
    return BeCa, BeCa_lo, BeCa_hi

print(get_BeCa())
print(get_BeCa(n_sigma=3.))

lodders_CI_LiCa=lodders_table.loc['Li']["A_el"]-lodders_table.loc['Ca']["A_el"]
print(lodders_CI_LiCa)

be/ca 0.004361702127659575 +/- 0.0020716256896796434
log10(be/ca): -2.3603439968799633 ,upper bound: -2.19156431807979 ,lower bound: -2.6401500215732696
(-2.3603439968799633, -2.6401500215732696, -2.19156431807979)
be/ca 0.004361702127659575 +/- 0.0020716256896796434
log10(be/ca): -2.3603439968799633 ,upper bound: -1.9756547742878527 ,lower bound: nan
(-2.3603439968799633, nan, -1.9756547742878527)
-2.9999999999999996


/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:64: RuntimeWarning: invalid value encountered in log10
  log_lo_bound=np.log10(lower_bound)


In [11]:
def get_spalled_LiCa(n_sigma=1.,cross_source=default_cross_source, cross_method='max',spall_product='Li',projectile='p',spall_target='O16'):
    """
    see pages 30-32 of General Clemens XI for algebra for calculation
    """
    BeCa, BeCa_lo, BeCa_hi=get_BeCa(n_sigma=n_sigma)
    BeCa_array=np.array([BeCa,BeCa_lo,BeCa_hi])
    
    
    if cross_source=='crude':
        if spall_product=='Li':
            Li7_cross_section=5e-26 #Li-7 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            Li6_cross_section=2e-26 #Li-6 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            Be9_cross_section=1e-26 #Be-9 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            LiCa_array=np.log10((Li7_cross_section+Li6_cross_section)/Be9_cross_section*10.**BeCa_array+10.**lodders_CI_LiCa)
        else:
            print('cross_source=="crude" only works for lithium (Li) as the product')
    elif cross_source=='read':
        energy_range=np.linspace(1,1000,1000)
        Be9_cross_section_array=nc.get_cross_section(energy_range, projectile=projectile, target=spall_target,product='A9')
        if cross_method=='max':
            max_energy_index=np.nanargmax(Be9_cross_section_array)
            max_cross_energy=energy_range[max_energy_index]
            print('Max Cross Section energy:',max_cross_energy, 'MeV')
            Be9_cross_section=Be9_cross_section_array[max_energy_index]
            if spall_product=='Li':
                Li6_cross_section=nc.get_cross_section(max_cross_energy, projectile=projectile, target=spall_target,product='A6')
                Li7_cross_section=nc.get_cross_section(max_cross_energy, projectile=projectile, target=spall_target,product='A7')
                print('Be-9 Cross Section:',Be9_cross_section)
                print('Li-6 Cross Section:', Li6_cross_section)
                print('Li-7 Cross Section:',Li7_cross_section)
                LiCa_array=np.log10((Li7_cross_section+Li6_cross_section)/Be9_cross_section*10.**BeCa_array+10.**lodders_CI_LiCa)
            elif spall_product=='B':
                print('spall_product==B not implemented yet')
            else:
                print('spall_product not recognized:', spall_product)
        else:
            print("cross_method not recognized:", cross_method)

    print('LiCa_array',LiCa_array)
    return LiCa_array

In [12]:
def plot_wd_errorbar(x_coord, el1el2, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label='', color='b'):
    if label=='':
        #label=name #commented this on 2021-11-12 to try to keep star points out of legend
        pass 
    else:
        pass
    uplims=False
    lolims=False
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el1el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(x_coord, el1el2, yerr= el1el2_err, uplims=uplims, lolims=lolims,  color=color,marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(x_coord,el1el2, label=label, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [13]:
def plot_CI_chondrite(x_coord,el1,el2=base_el,label='',show=True):
    el1_val=lodders_table.loc[el1]['A_el']
    el2_val=lodders_table.loc[el2]['A_el']
    el1_err=lodders_table.loc[el1]['A_el_err']
    el2_err=lodders_table.loc[el2]['A_el_err']
    el1el2=el1_val-el2_val
    el1el2_err=np.sqrt(el1_err**2+el2_err**2)
    if show:
        plt.errorbar(x_coord, el1el2, yerr= el1el2_err,  color=met_color,marker=met_marker,  markersize=ci_size,linestyle='None', label=label)
    else:
        pass
    return el1el2

In [14]:
spt.initiate_science_plot()
spt.start_ApJ_fig(width_cols=2,constrained_layout=True,width_height=[1,0.5]) #before adding decreasing phase arrows
spt.start_ApJ_fig(width_cols=2,constrained_layout=True,width_height=[1,0.5/(2+3.5)*(2.75+4.5)]) #attempting to keep the relative spacing the same as before even though I've expanded the y-range.

#plt.figure(figsize=(10.,7.25))
ax=plt.subplot()



for i,name in enumerate(el_list):
    print(i,name)
    edge_spot=i*el_space
    plt.axvline(x=edge_spot, linestyle=':',color='k')
    num_wds=len(use_wd_abund_table)
    num_pop=len(use_pop_table)
    #tot_objects= num_wds+2 #this is for the future when I include CI Chondrites, continental crust, etc. + number of non-WD points
    tot_objects= num_wds+num_pop+2 #this is for the future when I include CI Chondrites, continental crust, etc. + number of non-WD points
    object_spacing=el_space/(tot_objects+1)
    arrow_width=el_space/(tot_objects+1)*0.9
    start_pos=edge_spot
    el_string=name.lower()+'/'+base_el.lower()
    original_el_string=el_string
    print('edge_spot+object_spacing*2.5',edge_spot+object_spacing*2.5)
    if name=='Li':
        plt.text(np.mean([edge_spot, edge_spot+object_spacing*2.5]),text_y_pos,'Solar System',rotation=90,va=text_va,ha='center',bbox=props)
        plt.text(np.mean([edge_spot+object_spacing*2.5, edge_spot+object_spacing*2.5+(num_pop)*object_spacing]),text_y_pos,'Main-Sequence Stars',rotation=90,va=text_va,ha='center',bbox=props)
        plt.text(np.mean([edge_spot+object_spacing*2.5+(num_pop)*object_spacing, edge_spot+el_space]),text_y_pos,'White Dwarfs',rotation=90,va=text_va,ha='center',bbox=props)


    plt.axvspan(edge_spot, edge_spot+object_spacing*2.5, alpha=0.1, edgecolor='k',hatch='\\\\',color='k',fill=False, linewidth=0)
    plt.axvspan(edge_spot+object_spacing*2.5, edge_spot+object_spacing*2.5+(num_pop)*object_spacing, alpha=0.1, edgecolor='k',hatch='...',color='k',fill=False,linewidth=0)
    plt.axvspan(edge_spot+object_spacing*2.5+(num_pop)*object_spacing, edge_spot+el_space, alpha=0.1, edgecolor='k',hatch='//',color='k',fill=False,linewidth=0)
    if i==0:
        CI_label='CI Chond.'
        crust_label='Cont. Crust'
    else:
        CI_label=''
        crust_label=''
    CI_el1el2=plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label,show=True)
    if name=='Li':
        J2339_LiCa_array=get_spalled_LiCa(n_sigma=spall_n_sigma)
        #plt.axhspan(J2339_LiCa_array[1],J2339_LiCa_array[2],xmin=0, xmax=1./len(el_list), alpha=0.1, color='k',linewidth=0)
        #plt.axhline(y=J2339_LiCa_array[0], xmin=0, xmax=1./len(el_list), linestyle='--',color='k')
    else:
        pass
    start_pos=start_pos+object_spacing #because we plotted the CI chondrite point
    plt.errorbar(start_pos+object_spacing,crust_table[el_string],color=met_color,marker=r'$\oplus$',markersize=ci_size+2,linestyle='None',label=crust_label,markeredgewidth=0.5)
    #to do CI-normalized:
    #plt.errorbar(start_pos+object_spacing,crust_table[el_string]-CI_el1el2,color=met_color,marker=r'$\oplus$',markersize=ci_size+2,linestyle='None',label=crust_label)
    
    start_pos=start_pos+object_spacing #because I also just plotted the continental crust point
    for j,row in enumerate(use_pop_table):
        object_pos=start_pos+j*object_spacing+object_spacing
        if i==0:
            label=row['name']
        else:
            label=''
        #have to add CI_el1el2 to the values in the SAGA pop table because I recorded the [el/Ca] values instead of log(el/Ca) values.
        #not any more I don't!
        if name=='Li':
            #print('log(Li/Ca)=',ALi_BBN-row['ca/h']-lodders_table.loc['Ca']['A_el'],np.sqrt(row['ca/h_err']**2.+lodders_table.loc['Ca']['A_el_err']**2.))
            #plot_wd_errorbar(object_pos, ALi_BBN-row['ca/h']-lodders_table.loc['Ca']['A_el'],row['ca/h_err'],row['name'],color=row['plot_color'],selected_marker=met_marker,markersize=ci_size, label=label)
            plot_wd_errorbar(object_pos, row[el_string],row[el_string+'_err'],row['name'],color=row['plot_color'],selected_marker=star_marker,markersize=ci_size, label=label)
        else:
            #plot_wd_errorbar(object_pos, row[el_string]+CI_el1el2,row[el_string+'_err'],row['name'],color=row['plot_color'],selected_marker=met_marker,markersize=ci_size, label=label)
            plot_wd_errorbar(object_pos, row[el_string],row[el_string+'_err'],row['name'],color=row['plot_color'],selected_marker=star_marker,markersize=ci_size, label=label)
    start_pos=start_pos+object_spacing*num_pop
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size
    for j,row in enumerate(use_wd_abund_table):
        object_pos=start_pos+j*object_spacing+object_spacing
        try:
            if i==0:
                label=row['display_name']
            else:
                label=''
            plot_wd_errorbar(object_pos, row[el_string],row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize, label=label)
            dp_row=dp_arrow_table.loc[row['name']]
            #print('dp_row',dp_row)
            if ((row['show_dp']) and (show_dp) and (original_el_string!='k/ca')):
                #excludes the K/Ca decreasing phase arrows because those are too short to get out from behind the symbols.
                arrow_head_width=arrow_width
                if (np.abs(dp_row['dp_'+original_el_string]) < arrow_head_length_mult*arrow_head_width):
                    print(el_string,'arrow_head_length would have been less than the arrow length...',dp_row['dp_'+original_el_string] ,"<",arrow_head_length_mult*arrow_head_width,row['show_dp'])
                    arrow_head_length=dp_row['dp_'+original_el_string]
                else:
                    arrow_head_length=arrow_head_length_mult*arrow_head_width
                plt.arrow(object_pos,row[el_string],0,dp_row['dp_'+original_el_string],color=row['plot_color'],width=arrow_width, alpha=arrow_alpha,length_includes_head=True,linewidth=0,head_width=arrow_head_width )
                #print(el_string,row['name'],object_pos,row[el_string],object_pos,dp_row['dp_'+original_el_string])
            else:
                print('show_dp is false',row['show_dp'], 'so not adding arrow')

            #below is the CI Chondrite-normalized version
            #plot_wd_errorbar(object_pos, row[el_string]-CI_el1el2,row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize, label=label)
        except KeyError as error:
            print("KeyError:",error)
    #plt.text(edge_spot+0.5*el_space,0,name)
    
    
#plt.ylabel('log(Z/Ca)')
#plt.ylabel('log(Z/Ca)-log(Z/Ca)_CI')
plt.xlim(0,len(el_list)*el_space)

if ssp:
    #plt.ylabel(r'$\log(\mathrm{Z/Ca})_{\mathrm{SSP}}$')
    plt.ylabel(r'$\log(\mathrm{el/Ca})_{\mathrm{SSP}}$')


else:
    #plt.ylabel(r'$\log(\mathrm{Z/Ca})_{\mathrm{IP}}$')
    plt.ylabel(r'$\log(\mathrm{el/Ca})_{\mathrm{IP}}$')


#plt.axhline(y=0, linestyle='--', color='k')
#plt.ylim(-3.5,2) #prior to adding the decreasing phase arrows
plt.ylim(-4.5,2.75)
x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
print(x_ticks)
ax.set_xticks(x_ticks)
ax.set_xticklabels(el_list)
#plt.legend(loc='best',fontsize=7)
plt.legend(loc='lower right',fontsize=7)


if savefig:
    print(os.getcwd())
    os.chdir(figure_output_dir)
    print(os.getcwd())
    start = time.time()
    print(start)
    time_string=str(start).split('.')[0]
    if ssp:
        plt.savefig("all_elements_ssp"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    else:
        plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    print("Figure saved")
else:
    pass
plt.show()

0 Li
edge_spot+object_spacing*2.5 0.38461538461538464
be/ca 0.004361702127659575 +/- 0.0020716256896796434
log10(be/ca): -2.3603439968799633 ,upper bound: -2.19156431807979 ,lower bound: -2.6401500215732696
Getting cross section of p + O16 -> A9
Max Cross Section energy: 65.0 MeV
Getting cross section of p + O16 -> A6
Getting cross section of p + O16 -> A7
Be-9 Cross Section: 10.0
Li-6 Cross Section: 28.0
Li-7 Cross Section: 55.0
LiCa_array [-1.42943222 -1.69880426 -1.26442808]


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:30: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:31: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:32: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.


show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
1 Na
edge_spot+object_spacing*2.5 2.3846153846153846
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
2 Mg
edge_spot+object_spacing*2.5 4.384615384615385
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


show_dp is false 1 so not adding arrow
3 K
edge_spot+object_spacing*2.5 6.384615384615385
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
4 Cr
edge_spot+object_spacing*2.5 8.384615384615385
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
5 Fe
edge_spot+object_spacing*2.5 10.384615384615385
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 0 so not adding arrow
show_dp is false 1 so not adding arrow
show_dp is false 1 so not adding arrow
[ 1.  3.  5.  7.  9. 11.]
/Users/BenKaiser/Desktop/radial_velocity_calculations
/Users/BenKaiser/Deskto

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:144: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [15]:
print(lodders_table['element'])

element
-------
     Li
     Be
     Na
     Mg
      K
     Ca
     Cr
     Fe


In [16]:
print(matplotlib.__version__)

3.3.4
